# Phase 27 — TextCNN

Υλοποίηση του TextCNN (Kim 2014) για Food Hazard Detection.

**Αρχιτεκτονική:**
```
Text → Embedding Layer → Conv1D (multiple filter sizes) → MaxPooling → Dense → Output
```

**Filter sizes:** [2, 3, 4] — κάθε filter μαθαίνει διαφορετικά n-grams

**Γιατί CNN για text:**
- Μαθαίνει τοπικά patterns (n-grams) αυτόματα
- Πολύ πιο γρήγορο από transformers
- Κλασική baseline μέθοδος για σύγκριση

**Σκοπός:** Σύγκριση με TF-IDF methods και transformers για την αναφορά.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from collections import Counter
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()

all_hazard  = pd.concat([train['hazard-category'],  valid['hazard-category']])
all_product = pd.concat([train['product-category'], valid['product-category']])
le_hazard.fit(all_hazard)
le_product.fit(all_product)

y_train_hazard  = le_hazard.transform(train['hazard-category'])
y_train_product = le_product.transform(train['product-category'])
y_valid_hazard  = le_hazard.transform(valid['hazard-category'])
y_valid_product = le_product.transform(valid['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')
print(f'Hazard classes: {n_hazard} | Product classes: {n_product}')

In [ ]:
# Vocabulary από train corpus
MAX_VOCAB = 30000
MAX_LEN   = 128
PAD_IDX   = 0
UNK_IDX   = 1

def tokenize(text):
    return text.lower().split()

# Build vocabulary από train
counter = Counter()
for text in texts_train:
    counter.update(tokenize(text))

# Κράτα τις MAX_VOCAB πιο συχνές λέξεις
vocab = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

print(f'Vocabulary size: {len(vocab):,}')

def text_to_ids(text, max_len=MAX_LEN):
    tokens = tokenize(text)[:max_len]
    ids    = [vocab.get(t, UNK_IDX) for t in tokens]
    # Padding
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

# Μετατροπή όλων των texts σε IDs
X_train_ids = [text_to_ids(t) for t in texts_train]
X_valid_ids = [text_to_ids(t) for t in texts_valid]
X_test_ids  = [text_to_ids(t) for t in texts_test]
print('Texts converted to IDs!')

In [ ]:
class TextDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = torch.tensor(ids,    dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], self.labels[idx]


# TextCNN — Kim (2014)
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_classes,
                 filter_sizes=[2, 3, 4], n_filters=128, dropout=0.5):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # Parallel Conv1D layers με διαφορετικά filter sizes
        # Κάθε filter size μαθαίνει διαφορετικά n-grams
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim,
                      out_channels=n_filters,
                      kernel_size=fs)
            for fs in filter_sizes
        ])

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_filters * len(filter_sizes), n_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)          # (batch, seq_len, embed_dim)
        embedded = embedded.permute(0, 2, 1)  # (batch, embed_dim, seq_len)

        # Conv + ReLU + MaxPool για κάθε filter size
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(embedded))        # (batch, n_filters, seq_len - fs + 1)
            c = F.max_pool1d(c, c.shape[2])  # (batch, n_filters, 1) — global max pooling
            pooled.append(c.squeeze(2))       # (batch, n_filters)

        # Concatenation όλων των filter outputs
        cat = torch.cat(pooled, dim=1)        # (batch, n_filters * len(filter_sizes))
        cat = self.dropout(cat)
        return self.classifier(cat)           # (batch, n_classes)


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'  Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

print('TextCNN architecture defined!')

In [ ]:
# Hyperparameters
EMBED_DIM    = 100
N_FILTERS    = 128
FILTER_SIZES = [2, 3, 4]
DROPOUT      = 0.5
BATCH_SIZE   = 64
LR           = 1e-3
N_EPOCHS     = 20
PATIENCE     = 3

print(f'Hyperparameters:')
print(f'  Embedding dim:  {EMBED_DIM}')
print(f'  Filter sizes:   {FILTER_SIZES}')
print(f'  N filters:      {N_FILTERS}')
print(f'  Batch size:     {BATCH_SIZE}')
print(f'  LR:             {LR}')
print(f'  Max epochs:     {N_EPOCHS}')

In [ ]:
import copy

def train_cnn(X_train_ids, y_train, X_valid_ids, y_valid, n_classes, task_name):
    print(f'\n=== ΕΚΠΑΙΔΕΥΣΗ TextCNN — {task_name} ===')

    train_loader = DataLoader(
        TextDataset(X_train_ids, y_train),
        batch_size=BATCH_SIZE, shuffle=True
    )
    valid_loader = DataLoader(
        TextDataset(X_valid_ids, y_valid),
        batch_size=BATCH_SIZE, shuffle=False
    )

    model = TextCNN(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        n_classes=n_classes,
        filter_sizes=FILTER_SIZES,
        n_filters=N_FILTERS,
        dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    best_f1      = 0
    best_state   = None
    patience_cnt = 0

    for epoch in range(N_EPOCHS):
        # Training
        model.train()
        total_loss = 0
        for ids, labels in train_loader:
            ids, labels = ids.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(ids), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        all_preds = []
        with torch.no_grad():
            for ids, labels in valid_loader:
                preds = model(ids.to(device)).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)

        f1 = f1_score(y_valid, all_preds, average='macro', zero_division=0)
        print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Valid F1: {f1:.4f}', end='')

        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())
            patience_cnt = 0
            print(' ✅')
        else:
            patience_cnt += 1
            print(f' (patience {patience_cnt}/{PATIENCE})')
            if patience_cnt >= PATIENCE:
                print(f'Early stopping!')
                break

    model.load_state_dict(best_state)
    print(f'Καλύτερο F1: {best_f1:.4f}')
    return model


def get_predictions(model, ids_list):
    model.eval()
    loader = DataLoader(
        TextDataset(ids_list, [0]*len(ids_list)),
        batch_size=BATCH_SIZE, shuffle=False
    )
    all_preds = []
    with torch.no_grad():
        for ids, _ in loader:
            preds = model(ids.to(device)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

## Εκπαίδευση TextCNN

In [ ]:
# Εκπαίδευση Hazard classifier
model_hazard = train_cnn(
    X_train_ids, y_train_hazard,
    X_valid_ids, y_valid_hazard,
    n_hazard, 'HAZARD'
)

In [ ]:
# Εκπαίδευση Product classifier
model_product = train_cnn(
    X_train_ids, y_train_product,
    X_valid_ids, y_valid_product,
    n_product, 'PRODUCT'
)

## Αξιολόγηση & Submission

In [ ]:
# Validation evaluation
pred_hazard_valid  = le_hazard.inverse_transform(get_predictions(model_hazard,  X_valid_ids))
pred_product_valid = le_product.inverse_transform(get_predictions(model_product, X_valid_ids))

print('=== ΑΞΙΟΛΟΓΗΣΗ TextCNN (validation) ===')
score_cnn = official_st1_score(
    valid['hazard-category'], pred_hazard_valid,
    valid['product-category'], pred_product_valid
)

print('\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Naive Bayes:       0.3122')
print(f'Random Forest:     0.4775')
print(f'MLP:               (δες phase25)')
print(f'TextCNN:           {score_cnn:.4f}')
print(f'Logistic Reg:      0.6523')
print(f'SVM (tuned):       0.7419')
print(f'DistilBERT:        0.7606')
print(f'Best Transformer:  0.8173')

In [ ]:
# Test predictions & submission
test_hazard  = le_hazard.inverse_transform(get_predictions(model_hazard,  X_test_ids))
test_product = le_product.inverse_transform(get_predictions(model_product, X_test_ids))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_textcnn.csv', index=False)

print(' Αποθηκεύτηκε: submission_textcnn.csv')
print(f'Validation score: {score_cnn:.4f}')